# RMSD Analysis — Binding Stability Comparison

Analyzes fixed trajectories and compares binding stability across candidates.

**Prerequisites:** Run the trajectory fix (mdtraj image_molecules + superpose) first for each candidate.

In [ ]:
!pip install -q mdtraj matplotlib
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# EDIT THIS LIST — add/remove candidates as needed
# ============================================================
CANDIDATES = {
    "R4_DPETGEDF": 50,      # name: ns_duration
    "NRF2_native": 80,
    # "R4_DPETGEFF": 80,    # uncomment when done
    # "R4_DPETGEWY": 80,
    # "R4_LDEETGEWY": 80,
}

BASE_DIR = "/content/drive/MyDrive/MD_Results"

In [ ]:
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt
import os

results = {}

for name, ns_total in CANDIDATES.items():
    base = f"{BASE_DIR}/{name}"
    fixed_dcd = f"{base}/{name}_fixed.dcd"
    protein_pdb = f"{base}/{name}_protein.pdb"

    if not os.path.exists(fixed_dcd):
        print(f"{name}: fixed files not found — run trajectory fix first")
        continue

    print(f"Loading {name}...")
    traj = md.load(fixed_dcd, top=protein_pdb)

    receptor_ca = traj.topology.select("chainid 0 and name CA")
    peptide_ca = traj.topology.select("chainid 1 and name CA")

    traj = traj.superpose(traj[0], atom_indices=receptor_ca)
    receptor_rmsd = md.rmsd(traj, traj[0], atom_indices=receptor_ca) * 10
    peptide_rmsd = md.rmsd(traj, traj[0], atom_indices=peptide_ca) * 10

    min_dists = []
    for i in range(traj.n_frames):
        pep_pos = traj.xyz[i, peptide_ca]
        rec_pos = traj.xyz[i, receptor_ca]
        diff = pep_pos[:, np.newaxis, :] - rec_pos[np.newaxis, :, :]
        dists = np.sqrt(np.sum(diff**2, axis=2))
        min_dists.append(np.min(dists) * 10)
    min_dists = np.array(min_dists)

    times = np.linspace(0, ns_total, traj.n_frames)
    equil = times >= ns_total / 2

    results[name] = {
        "peptide_rmsd_mean": np.mean(peptide_rmsd[equil]),
        "peptide_rmsd_std": np.std(peptide_rmsd[equil]),
        "contact_mean": np.mean(min_dists[equil]),
        "contact_std": np.std(min_dists[equil]),
        "receptor_rmsd_mean": np.mean(receptor_rmsd),
        "times": times,
        "peptide_rmsd": peptide_rmsd,
        "min_dists": min_dists,
        "receptor_rmsd": receptor_rmsd,
        "ns": ns_total,
    }

    contact = results[name]['contact_mean']
    if contact < 4.0:
        verdict = "STABLE BINDER"
    elif contact < 6.0:
        verdict = "MARGINAL"
    else:
        verdict = "DISSOCIATED"

    print(f"\n{'='*50}")
    print(f"{name} ({ns_total} ns)")
    print(f"{'='*50}")
    print(f"  Receptor RMSD:      {np.mean(receptor_rmsd):.2f} A")
    print(f"  Peptide RMSD:       {results[name]['peptide_rmsd_mean']:.2f} +/- {results[name]['peptide_rmsd_std']:.2f} A")
    print(f"  Min contact:        {contact:.2f} +/- {results[name]['contact_std']:.2f} A")
    print(f"  VERDICT:            {verdict}")

print(f"\n\nLoaded {len(results)} candidates.")

In [ ]:
# ============================================================
# Summary Table
# ============================================================
print(f"{'Candidate':<20} {'Duration':<10} {'Rec RMSD':<12} {'Pep RMSD':<16} {'Min Contact':<16} {'Verdict'}")
print(f"{'-'*20} {'-'*10} {'-'*12} {'-'*16} {'-'*16} {'-'*15}")

for name, r in results.items():
    contact = r['contact_mean']
    if contact < 4.0:
        verdict = "STABLE"
    elif contact < 6.0:
        verdict = "MARGINAL"
    else:
        verdict = "DISSOCIATED"
    print(f"{name:<20} {r['ns']:<10} {r['receptor_rmsd_mean']:<12.2f} {r['peptide_rmsd_mean']:.2f}+/-{r['peptide_rmsd_std']:.2f}{'':>3} {r['contact_mean']:.2f}+/-{r['contact_std']:.2f}{'':>3} {verdict}")

In [ ]:
# ============================================================
# Plots
# ============================================================
n = len(results)
if n == 0:
    print("No results to plot.")
else:
    fig, axes = plt.subplots(2, n, figsize=(7*n, 8), squeeze=False)

    for idx, (name, r) in enumerate(results.items()):
        axes[0][idx].plot(r['times'], r['peptide_rmsd'], 'b-', linewidth=0.5)
        axes[0][idx].axhline(y=3, color='g', linestyle='--', alpha=0.5, label='3 A')
        axes[0][idx].axhline(y=5, color='r', linestyle='--', alpha=0.5, label='5 A')
        axes[0][idx].set_title(f"{name} — Peptide RMSD")
        axes[0][idx].set_ylabel("RMSD (A)")
        axes[0][idx].set_ylim(0, max(10, np.max(r['peptide_rmsd']) * 1.1))
        axes[0][idx].legend(fontsize=8)

        axes[1][idx].plot(r['times'], r['min_dists'], 'r-', linewidth=0.5)
        axes[1][idx].axhline(y=4, color='g', linestyle='--', alpha=0.5, label='Bound (4 A)')
        axes[1][idx].set_title(f"{name} — Min Contact")
        axes[1][idx].set_xlabel("Time (ns)")
        axes[1][idx].set_ylabel("Distance (A)")
        axes[1][idx].set_ylim(0, max(10, np.max(r['min_dists']) * 1.1))
        axes[1][idx].legend(fontsize=8)

    plt.suptitle("Binding Stability Comparison", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{BASE_DIR}/binding_comparison.png", dpi=150)
    plt.show()
    print(f"\nPlot saved: {BASE_DIR}/binding_comparison.png")

## Fix Trajectory (run this if _fixed.dcd doesn't exist for a candidate)

Change `NAME` and run the cell below. Then re-run the analysis cells above.

In [ ]:
# ============================================================
# Fix trajectory for a single candidate
# Change NAME to the candidate you want to fix
# ============================================================
NAME = "R4_DPETGEFF"  # CHANGE THIS
NS = 80               # CHANGE THIS to match simulation length

BASE = f"{BASE_DIR}/{NAME}"

print(f"Loading {NAME} trajectory...")
traj = md.load(f"{BASE}/{NAME}_trajectory.dcd", top=f"{BASE}/{NAME}_solvated.pdb")
print(f"Loaded: {traj.n_frames} frames")

print("Fixing PBC...")
traj = traj.image_molecules()

print("Aligning on receptor...")
receptor_atoms = traj.topology.select("chainid 0 and name CA")
traj = traj.superpose(traj[0], atom_indices=receptor_atoms)

print("Saving protein-only trajectory...")
protein_atoms = traj.topology.select("protein")
protein = traj.atom_slice(protein_atoms)
protein.save(f"{BASE}/{NAME}_fixed.dcd")
protein[0].save(f"{BASE}/{NAME}_protein.pdb")

print(f"Done: {protein.n_frames} frames, {protein.n_atoms} atoms")
print(f"Now re-run the analysis cells above.")